# CDSM Thesis Reproduction (Public)

This notebook summarizes the commands needed to reproduce the thesis experiments
based on `main.tex` and the current `conditional_fourier_diffusion` code.

Assume you are running from the `conditional_fourier_diffusion_public` repo root.

## 0. Environment Setup

In [ ]:
# Main environment (cfdiff/fdiff)
conda create -n cfdiff python=3.10 -y
conda activate cfdiff
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install -e .
pip install pot ipywidgets lightning[extra] gluonts pandas matplotlib seaborn requests PyYAML tqdm pillow

In [ ]:
# Baseline environment (econometrics + DeepVAR)
conda create -n baselines python=3.10 -y
conda activate baselines
pip install arch PyPortfolioOpt statsmodels gluonts pandas matplotlib seaborn

## 1. Project Path and Environment Variables

In [ ]:
cd /path/to/conditional_fourier_diffusion_public
conda activate cfdiff

# Optional: reduce Fourier metric cost
export CFDIFF_SW_DIRECTIONS=64
export CFDIFF_METRICS_SAMPLE_LIMIT=64
export CFDIFF_SINKHORN_METHOD=epsilon_scaling
export CFDIFF_SINKHORN_REG=1e-2
# If VRAM is tight, temporarily disable Fourier metrics
# export CFDIFF_DISABLE_FOURIER=1

# Required for scripts that import src/
export PYTHONPATH=src

## 2. Dataset Preparation (matching thesis time ranges)

Time ranges are taken from `assets/data_overview.tex`.

### 2.1 Exchange (GluonTS exchange_rate subset)
- Train: 1995-01-04 to 1998-10-09
- Test: 1998-10-13 to 1999-09-20
- Assets: 8
- Windows: C=30, P=15

In [ ]:
python scripts/prepare_exchange_rate_dataset.py   --dst-dir ~/.gluonts/datasets/exchange_rate_clean   --train-ratio 0.8   --trim-start 1995-01-04   --trim-end 1999-09-20   --nan-policy none

### 2.2 FX29 (ECB FX30 with USD removed)
- Train: 2020-01-03 to 2024-08-27
- Test: 2024-09-24 to 2025-11-21
- Assets: 29 (`--exclude-usd`)
- Windows: C=45, P=15

In [ ]:
python scripts/prepare_fx30_ecb_dataset.py   --dst-dir ~/.gluonts/datasets/fx30_ecb   --trim-start 2020-01-03   --trim-end 2025-11-21   --train-ratio 0.8   --returns log   --prediction-length 15   --exclude-usd   --nan-policy none   --overwrite

### 2.3 Industry49 (Ken French 49 Industry Portfolios)
- Train: 2018-01-02 to 2022-10-18
- Test: 2022-10-19 to 2023-12-29
- Assets: 49
- Windows: C=30, P=15

In [ ]:
python scripts/prepare_industry49_dataset.py   --dst-dir ~/.gluonts/datasets/industry49_clean   --train-ratio 0.8   --trim-start 2018-01-02   --trim-end 2023-12-29   --nan-policy none   --overwrite

### 2.4 iShares14 (Yahoo Finance)
- Train: 2015-11-13 to 2019-02-11
- Test: 2019-04-01 to 2020-01-21
- Assets: 14
- Windows: C=30, P=15 (short) and C=100, P=20 (medium)

You need a local CSV (the repo includes `ishares/df_ishares14x1102.csv`).

In [ ]:
python scripts/prepare_ishares_dataset.py   --src-file ishares/df_ishares14x1102.csv   --dst-dir ~/.gluonts/datasets/ishares14_clean   --train-ratio 0.8   --trim-start 2015-11-13   --trim-end 2020-01-21   --returns log   --overwrite

## 3. Dataset Overview + Window Counts (Table 1)

In [ ]:
python scripts/make_dataset_summary.py   --entry "Exchange|~/.gluonts/datasets/exchange_rate_clean|train|30|15|0.3|GluonTS Exchange Rate|1995-1999|8"   --entry "FX29|~/.gluonts/datasets/fx30_ecb|train|45|15|0.3|ECB FX (USD base)|2020-2025|29"   --entry "Industry|~/.gluonts/datasets/industry49_clean|train|30|15|0.3|Ken French Industry 49|2018-2023|49"   --entry "iShares|~/.gluonts/datasets/ishares14_clean|train|30|15|0.3|iShares ETFs|2015-2020|14"   --entry "iShares|~/.gluonts/datasets/ishares14_clean|train|100|20|0.3|iShares ETFs (medium)|2015-2020|14"   --output-markdown assets/dataset_summary.md   --output-latex assets/dataset_summary.tex

python scripts/make_window_count_table.py   --dataset Exchange:~/.gluonts/datasets/exchange_rate_clean:30:15:0.3:0.8:1   --dataset FX29:~/.gluonts/datasets/fx30_ecb:45:15:0.3:0.8:1   --dataset Industry:~/.gluonts/datasets/industry49_clean:30:15:0.3:0.8:1   --dataset iShares:~/.gluonts/datasets/ishares14_clean:30:15:0.3:0.8:1   --dataset iShares:~/.gluonts/datasets/ishares14_clean:100:20:0.3:0.8:1   --output-markdown assets/window_counts.md   --output-latex assets/window_counts.tex

python scripts/make_data_overview_table.py   --dataset-summary assets/dataset_summary.tex   --window-counts assets/window_counts.tex   --out assets/data_overview.tex

## 4. Training and Sampling (CDSM / UCDSM)

Hyperparameters from `tab:hyperparams` in the thesis:
- Transformer: d_model=192, layers=6, heads=8, dropout=0
- Optimizer: AdamW, lr=5e-4, weight_decay=0
- LR schedule: cosine warmup (step)
- Timestep sampling: Beta(2,5) + importance correction
- Epochs: 80
- Sampling steps: 1000
- Sampling batch size: 64
- Sampling repeats: 5

Use the templates below for each dataset (Exchange / FX29 / Industry / iShares14).

### 4.1 CDSM-T (time domain)

In [ ]:
# Example: Exchange (C=30, P=15)
export CFDIFF_DATA_DIR=~/.gluonts/datasets/exchange_rate_clean

python -m cfdiff.cmd.train   experiment=time   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.val_ratio=0.3   experiment.datamodule.stride=1   experiment.model.d_model=192   experiment.model.num_layers=6   experiment.model.n_head=8   experiment.model.dropout=0.0   experiment.model.lr=5e-4   experiment.model.weight_decay=0.0   experiment.model.lr_scheduler_type=cosine_warmup   experiment.model.t_sampling_mode=beta   experiment.model.t_beta_alpha=2   experiment.model.t_beta_beta=5   experiment.model.t_importance_correction=true   experiment.trainer.max_epochs=80   callbacks.epoch_metrics.enabled=true   callbacks.epoch_metrics.compute_fourier=true

python -m cfdiff.cmd.sample   experiment=time   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.val_ratio=0.3   experiment.sampler.num_diffusion_steps=1000   experiment.sampler.sample_batch_size=64   experiment.sampler.sample_repeats=5   checkpoint_path=outputs/time/conditional/<RUN>/model.ckpt   output_dir=outputs/time/conditional/<RUN>   run_mode=sample

### 4.2 CDSM-S (frequency domain)

In [ ]:
# Same overrides as CDSM-T, but use experiment=fourier
python -m cfdiff.cmd.train   experiment=fourier   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.val_ratio=0.3   experiment.datamodule.stride=1   experiment.model.d_model=192   experiment.model.num_layers=6   experiment.model.n_head=8   experiment.model.dropout=0.0   experiment.model.lr=5e-4   experiment.model.weight_decay=0.0   experiment.model.lr_scheduler_type=cosine_warmup   experiment.model.t_sampling_mode=beta   experiment.model.t_beta_alpha=2   experiment.model.t_beta_beta=5   experiment.model.t_importance_correction=true   experiment.trainer.max_epochs=80   callbacks.epoch_metrics.enabled=true   callbacks.epoch_metrics.compute_fourier=true

python -m cfdiff.cmd.sample   experiment=fourier   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.val_ratio=0.3   experiment.sampler.num_diffusion_steps=1000   experiment.sampler.sample_batch_size=64   experiment.sampler.sample_repeats=5   checkpoint_path=outputs/fourier/conditional/<RUN>/model.ckpt   output_dir=outputs/fourier/conditional/<RUN>   run_mode=sample

### 4.3 UCDSM (unconditional, fdiff)

In [ ]:
# UCDSM-T (time)
python -m fdiff.cmd.train   experiment=unconditional   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.fourier_transform=false   experiment.datamodule.val_ratio=0.3   experiment.model.d_model=192   experiment.model.num_layers=6   experiment.model.n_head=8   experiment.model.lr_max=5e-4   experiment.model.t_sampling_mode=beta   experiment.model.t_beta_alpha=2   experiment.model.t_beta_beta=5   experiment.model.t_importance_correction=true   experiment.trainer.max_epochs=80

python -m fdiff.cmd.sample   experiment=unconditional   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.fourier_transform=false   experiment.datamodule.val_ratio=0.3   experiment.sampler.num_diffusion_steps=1000   experiment.sampler.sample_batch_size=64   experiment.sampler.sample_repeats=5   checkpoint_path=outputs/time/unconditional/<RUN>/model.ckpt   output_dir=outputs/time/unconditional/<RUN>   run_mode=sample

# UCDSM-S (fourier)
python -m fdiff.cmd.train   experiment=unconditional   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.fourier_transform=true   experiment.datamodule.val_ratio=0.3   experiment.model.d_model=192   experiment.model.num_layers=6   experiment.model.n_head=8   experiment.model.lr_max=5e-4   experiment.model.t_sampling_mode=beta   experiment.model.t_beta_alpha=2   experiment.model.t_beta_beta=5   experiment.model.t_importance_correction=true   experiment.trainer.max_epochs=80

python -m fdiff.cmd.sample   experiment=unconditional   experiment.datamodule.data_dir=$CFDIFF_DATA_DIR   experiment.datamodule.context_len=30   experiment.datamodule.pred_len=15   experiment.datamodule.fourier_transform=true   experiment.datamodule.val_ratio=0.3   experiment.sampler.num_diffusion_steps=1000   experiment.sampler.sample_batch_size=64   experiment.sampler.sample_repeats=5   checkpoint_path=outputs/fourier/unconditional/<RUN>/model.ckpt   output_dir=outputs/fourier/unconditional/<RUN>   run_mode=sample

## 5. Baselines (Sample Cov / Ledoit-Wolf / RiskMetrics / DCC-GARCH / DeepVAR / CAB)

In [ ]:
# Example: Sample Covariance (run inside baselines env)
conda run -n baselines python src/baselines/cmd/run.py   +experiment=time   datamodule.data_dir=$CFDIFF_DATA_DIR   datamodule.context_len=30 datamodule.pred_len=15   datamodule.stride=1 datamodule.val_ratio=0.3 datamodule.batch_size=32   datamodule.num_workers=8   baseline.method=sample_cov   baseline.include_val=true   baseline.window_index=-1   baseline.save_csv=true baseline.save_pt=true

# Other methods: ledoit_wolf / riskmetrics / ewma / ccc_garch / cab

In [ ]:
# DeepVAR / DeepAR baselines (baselines env)
conda run -n baselines python scripts/run_deepar_deepvar_baselines.py   --data-dir $CFDIFF_DATA_DIR   --context-len 30 --pred-len 15   --val-ratio 0.3   --model deepvar

## 6. Metrics, Tables, and Figures

In [ ]:
# Main metric tables (example: structural metrics)
python scripts/make_results_table.py   --outputs-root outputs   --metrics     matrix_cov_fro matrix_cov_fro_rel matrix_cov_mae matrix_cov_mse matrix_cov_diag_mape     matrix_corr_fro matrix_corr_fro_rel matrix_corr_cross_mse     matrix_corr_offdiag_pearson matrix_corr_offdiag_spearman matrix_corr_sign_rate   --include-epoch   --table-csv assets/best_metrics_struct.csv   --table-tex assets/best_metrics_struct.tex

# Context augmentation / ablation tables
python scripts/make_ablation_context_table.py --help

In [ ]:
# Full paper figure batch
bash scripts/plot_paper_figures.sh

## 7. Notes
- iShares data requires a local CSV under `ishares/`.
- For long runs, keep separate output directories per dataset/config.
- For a complete command log, see `notebooks/paper.ipynb`.